#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.window import Window
from pyspark.sql.functions import trim, col, length

#Read Bronze Table

In [0]:
df = spark.table("demo_project.bronze.erp_px_cat_g1v2")

##Sanity Checks

In [0]:
df.limit(10).display()

#Silver Transformations

##Renaming Columns

In [0]:
rename_map = {
    "ID": "category_id",
    "CAT": "category",
    "SUBCAT": "sub_category",
    "maintenance": "maintenance_flag"
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity checks

In [0]:
df.limit(10).display()

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


##Normalize Maintenance Flag to Boolean

In [0]:
df = df.withColumn(
    "maintenance_flag",
    F.when(F.upper(col("maintenance_flag")) == "YES", F.lit(True))
    .when(F.upper(col("maintenance_flag")) == "NO", F.lit(False))
    .otherwise(None),
)

##Sanity Checks

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("demo_project.silver.erp_product_category")

##Sanity checks

In [0]:
%sql
select * from demo_project.silver.erp_product_category limit 10;